In [1]:
import pandas as pd
import plotly.express as px
import os

In [2]:
# 1. Definimos la ruta absoluta usando r"" para que Windows y Python se entiendan

ruta_ubicaciones = '../data/extra/202468-283-intensidad-trafico.csv'
# 2. Cargamos el archivo
df_ubicaciones = pd.read_csv(ruta_ubicaciones, sep=';', encoding='latin-1')

# 3. ¡Limpieza fundamental! Quitamos los espacios fantasma de las cabeceras
df_ubicaciones.columns = df_ubicaciones.columns.str.strip()

# Verificamos rápidamente las columnas limpias
print("Columnas listas:", df_ubicaciones.columns.tolist())

Columnas listas: ['tipo_elem', 'distrito', 'id', 'cod_cent', 'nombre', 'utm_x', 'utm_y', 'longitud', 'latitud']


In [3]:
# 2. Modifica estas dos variables SI Y SOLO SI los nombres impresos arriba son distintos
# (Ejemplo: si la columna se llama 'Latitud' con L mayúscula, cámbialo aquí)
col_lat = 'latitud' 
col_lon = 'longitud'

# Limpiamos nulos
df_ubicaciones = df_ubicaciones.dropna(subset=[col_lat, col_lon])


ids_m30 = [
7122
,7123
,1017
,1018
,1016
,1015
,1014
,1013
,1012
,6827
,6838
,6839
,6840
,6843
,6842
,6845
,6844
,11486
,6846
,6847
,6849
,6853
,6854
,6864
,6855
,1037
,1036
,1035
,6058
,7145
,7118
,7117
,6866
,1021
,6868
,1020
,1019
,6883
,6890
,6889
]  # <- pegas aquí los IDs que identificaste

##df_ubicaciones.loc[df_ubicaciones["id"].isin(ids_m30), "tipo_elem"] = "M30-Added"

# Creamos tipo_elem_aux con etiquetas (copia de tipo_elem pero con M30-Added)
df_ubicaciones["tipo_elem_aux"] = df_ubicaciones["tipo_elem"].copy()
df_ubicaciones.loc[df_ubicaciones["id"].isin(ids_m30), "tipo_elem_aux"] = "M30-Added"

# Creamos tipo_elem_m30 con 0, 1 y 2
df_ubicaciones["tipo_elem_m30"] = 0
df_ubicaciones.loc[df_ubicaciones["tipo_elem"] == "M30", "tipo_elem_m30"] = 1
df_ubicaciones.loc[df_ubicaciones["id"].isin(ids_m30), "tipo_elem_m30"] = 2

# Verificar
print(df_ubicaciones[["id", "tipo_elem", "tipo_elem_aux", "tipo_elem_m30"]].head(20))




       id tipo_elem tipo_elem_aux  tipo_elem_m30
0    6900     other         other              0
1    6668       M30           M30              1
2    6669       M30           M30              1
3    6688       M30           M30              1
4   11116       URB           URB              0
5    6691       M30           M30              1
6    6692       M30           M30              1
7    6899     other         other              0
8   10180       M30           M30              1
9    6914       M30           M30              1
10   6703       M30           M30              1
11   6716       M30           M30              1
12   6904     other         other              0
13   6905     other         other              0
14   6906     other         other              0
15   6719       M30           M30              1
16   6720       M30           M30              1
17   6722       M30           M30              1
18   6901     other         other              0
19   6732       M30 

In [4]:
# PASO PREVIO: Convertir coordenadas de texto a número
# Los archivos españoles usan coma como separador decimal (ej: "40,4168")
df_ubicaciones[col_lat] = df_ubicaciones[col_lat].astype(str).str.replace(',', '.').astype(float)
df_ubicaciones[col_lon] = df_ubicaciones[col_lon].astype(str).str.replace(',', '.').astype(float)

# Verificamos que ya son números
print(df_ubicaciones[[col_lat, col_lon]].dtypes)
print(df_ubicaciones[[col_lat, col_lon]].head(3))

latitud     float64
longitud    float64
dtype: object
     latitud  longitud
0  40.404237 -3.652823
1  40.417722 -3.660063
2  40.417468 -3.660344


In [5]:
fig = px.scatter_map(
    df_ubicaciones,
    lat=col_lat,
    lon=col_lon,
    hover_name="id",
    color="tipo_elem_aux",
    color_discrete_map={
        "M30":       "#FF0000",   # rojo
        "M30-Added": "#FF8C00",   # naranja
        "URB":       "#2ECC71",   # verde
        "other":     "#3498DB"    # azul
    },
    zoom=11,
    map_style="carto-positron",
    title="Ubicación de la Red de Sensores de Tráfico"
)

fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.show()
fig.show(renderer="browser")





In [6]:
ubicaciones_m30 = df_ubicaciones[df_ubicaciones["tipo_elem_m30"].isin([1, 2])].copy()

print(f"Total sensores M30: {len(ubicaciones_m30)}")
print(ubicaciones_m30["tipo_elem_aux"].value_counts())

Total sensores M30: 353
tipo_elem_aux
M30          314
M30-Added     39
Name: count, dtype: int64


In [7]:
# Guardamos archivo csv en la misma carpeta
print(ubicaciones_m30.head())
#ubicaciones_m30.to_csv('../data/extra/ubicaciones_m30.csv', index=False)


  tipo_elem  distrito    id cod_cent   nombre        utm_x        utm_y  \
1       M30       3.0  6668  PM10768  PM10768  444001,8459  4474331,118   
2       M30       3.0  6669  PM10766  PM10766   443977,812  4474303,082   
3       M30       2.0  6688  PM11303  PM11303  441272,9676  4470633,619   
5       M30       9.0  6691  PM12121  PM12121  437426,5051  4476616,897   
6       M30       9.0  6692  PM12211  PM12211  437273,1034  4477399,206   

   longitud    latitud tipo_elem_aux  tipo_elem_m30  
1 -3.660063  40.417722           M30              1  
2 -3.660344  40.417468           M30              1  
3 -3.691886  40.384225           M30              1  
5 -3.737787  40.437845           M30              1  
6 -3.739673  40.444881           M30              1  


In [8]:
fig_m30 = px.scatter_map(
    ubicaciones_m30,
    lat=col_lat,
    lon=col_lon,
    hover_name="id",
    color="tipo_elem_aux",
    color_discrete_map={
        "M30":       "#FF0000",   # rojo
        "M30-Added": "#FF8C00",   # naranja
    },
    zoom=12,
    map_style="carto-positron",
    title="Sensores de Tráfico M30"
)

fig_m30.update_traces(marker=dict(size=6, opacity=0.7))
fig_m30.show()
fig_m30.show(renderer="browser")